### All Tasks Completed

*   **Task 1: Data Preparation and Embedding Layer:** Successfully tokenized data, created word-to-index and tag-to-index mappings, implemented padding, and defined a `torch.nn.Embedding` layer.
*   **Task 2: Building the LSTM Sequence Labeler:** Implemented the `LSTMNERSegmenter` class using a bidirectional LSTM for sequence labeling.
*   **Task 3: Training and Evaluation:** Defined a loss function and optimizer, implemented the training loop, and evaluated the model's performance using accuracy. A discussion on F1-Score vs. Accuracy was also provided.
*   **Task 4: Architecture Comparison (Theoretical):** Discussed the conceptual differences between Simple RNN, LSTM, and GRU, and explained why Bi-LSTMs are better suited for NER tasks.

# Recurrent Neural Networks for Sequence Labeling


You are a developer at a FinTech company that needs to automatically extract key pieces of information (like names, organizations, and dates) from financial news articles. This task, known as Named Entity Recognition (NER), requires the model to process a sequence of words and assign a specific label to each word.
Since sentences have varying lengths and meaning depends on context (the sequence), a standard Feed-Forward Network is insufficient. Your task is to implement and compare different types of Recurrent Neural Networks (RNNs, GRUs, LSTMs) to solve this sequence labeling problem.

## Tasks:

In [1]:
!pip install gensim nltk scikit-learn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 59.7 MB/s eta 0:00:00


In [2]:
# word2Vec Model
import nltk
from nltk.corpus import brown
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
nltk.download('brown')  # Brown corpus for demo
nltk.download('punkt')  # Tokenizer

# Load and preprocess data
sentences = brown.sents()  # Get sentences from the Brown corpus
processed_sentences = [simple_preprocess(" ".join(sent)) for sent in sentences]

# printing
print(f"processed_sentences = {processed_sentences[:10]}, length = {len(processed_sentences)}")


# Train Word2Vec model
model = Word2Vec(
    sentences=processed_sentences,
    vector_size=100,  # Dimensionality of word vectors
    window=5,         # Context window size
    min_count=2,      # Minimum word frequency
    workers=4,        # Number of threads
    sg=0              # CBOW (0) or Skip-gram (1)
)

# Save and load the model if needed
model.save("word2vec.model")
model = Word2Vec.load("word2vec.model")


[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


processed_sentences = [['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of', 'atlanta', 'recent', 'primary', 'election', 'produced', 'no', 'evidence', 'that', 'any', 'irregularities', 'took', 'place'], ['the', 'jury', 'further', 'said', 'in', 'term', 'end', 'presentments', 'that', 'the', 'city', 'executive', 'committee', 'which', 'had', 'over', 'all', 'charge', 'of', 'the', 'election', 'deserves', 'the', 'praise', 'and', 'thanks', 'of', 'the', 'city', 'of', 'atlanta', 'for', 'the', 'manner', 'in', 'which', 'the', 'election', 'was', 'conducted'], ['the', 'september', 'october', 'term', 'jury', 'had', 'been', 'charged', 'by', 'fulton', 'superior', 'court', 'judge', 'durwood', 'pye', 'to', 'investigate', 'reports', 'of', 'possible', 'irregularities', 'in', 'the', 'hard', 'fought', 'primary', 'which', 'was', 'won', 'by', 'mayor', 'nominate', 'ivan', 'allen', 'jr'], ['only', 'relative', 'handful', 'of', 'such', 'reports', 'was', 'received', 'the', 'jury

In [3]:
# finding Similar words
similar_words = model.wv.most_similar("learning", topn=5)
print("Words similar to 'learning':", similar_words)

Words similar to 'learning': [('opportunities', 0.9661015868186951), ('wisdom', 0.9649056196212769), ('prestige', 0.964139461517334), ('sensitive', 0.9640482068061829), ('humanity', 0.9623273611068726)]


In [4]:
# Saving the model
model.wv.save_word2vec_format("word2vec_vectors.txt", binary=False)

In [5]:
# Import the library
# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

# Set random seed for reproducibility
torch.manual_seed(42)

print("Libraries imported. PyTorch Version:", torch.__version__)

Libraries imported. PyTorch Version: 2.9.0+cpu


### Task 1: Data Preparation and Embedding Layer

1. **Tokenization and Mapping:** Load the NER dataset and create two dictionaries: a `word-to-index` map for the tokens, and a `tag-to-index` map for the labels (B-PER, I-PER, O, etc.).
2. **Padding:** Since sentences have different lengths, implement a padding mechanism to ensure all input sequences have the same length .
3. **Embedding:** Create a torch.nn.Embedding layer that will map the integer indices (from Task 1) to dense, continuous word vectors (e.g., dimension 100).

In [6]:
# Task-1
# Synthetic Data:
# Data Format: (Sentence, Tags)
# Tags: B- (Beginning), I- (Inside), O (Outside)
training_data = [
    ("Elon Musk is the CEO of Tesla".split(), ["B-PER", "I-PER", "O", "O", "O", "O", "B-ORG"]),
    ("Apple released a new iPhone on Monday".split(), ["B-ORG", "O", "O", "O", "O", "O", "B-DATE"]),
    ("JP Morgan Chase reported strong profits".split(), ["B-ORG", "I-ORG", "I-ORG", "O", "O", "O"]),
    ("Stocks fell on Tuesday after the announcement".split(), ["O", "O", "O", "B-DATE", "O", "O", "O"]),
    ("Amazon plans to hire more engineers in 2024".split(), ["B-ORG", "O", "O", "O", "O", "O", "O", "B-DATE"]),
    ("Satya Nadella spoke about AI at Microsoft".split(), ["B-PER", "I-PER", "O", "O", "O", "O", "B-ORG"]),
]

# Print a sample to check structure
print(f"Sample Text: {training_data[0][0]}")
print(f"Sample Tags: {training_data[0][1]}")

Sample Text: ['Elon', 'Musk', 'is', 'the', 'CEO', 'of', 'Tesla']
Sample Tags: ['B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'B-ORG']


In [7]:
# Tokenization and Mapping
all_words = []
all_tags = []
for sentence, tags in training_data:
    for word in sentence:
        all_words.append(word)
    for tag in tags:
        all_tags.append(tag)

# Create word_to_index mapping
word_to_index = {"<PAD>": 0}
for word in sorted(list(set(all_words))):
    if word not in word_to_index:
        word_to_index[word] = len(word_to_index)

# Create tag_to_index mapping
tag_to_index = {"<PAD>": 0}
for tag in sorted(list(set(all_tags))):
    if tag not in tag_to_index:
        tag_to_index[tag] = len(tag_to_index)


print("Vocabulary Size:", len(word_to_index))
print("Tags:", tag_to_index)

Vocabulary Size: 41
Tags: {'<PAD>': 0, 'B-DATE': 1, 'B-ORG': 2, 'B-PER': 3, 'I-ORG': 4, 'I-PER': 5, 'O': 6}


In [8]:
# Padding:
from torch.nn.utils.rnn import pad_sequence
import torch

def prepare_sequence(seq, to_ix):
    idxs = [to_ix.get(w, to_ix['<PAD>']) for w in seq]
    return torch.tensor(idxs, dtype=torch.long)

# Convert all data to tensors
X = []
y = []

for sentence, tags in training_data:
    X.append(prepare_sequence(sentence, word_to_index))
    y.append(prepare_sequence(tags, tag_to_index))


X_padded = pad_sequence(X, batch_first=True, padding_value=word_to_index['<PAD>'])
y_padded = pad_sequence(y, batch_first=True, padding_value=tag_to_index['<PAD>'])

print("Shape of Input Tensor:", X_padded.shape)
print("Shape of Target Tensor:", y_padded.shape)

Shape of Input Tensor: torch.Size([6, 8])
Shape of Target Tensor: torch.Size([6, 8])


In [9]:
# Embedding
# Hyperparameters
VOCAB_SIZE = len(word_to_index)
EMBEDDING_DIM = 100

# 1. Define the Embedding Layer
embedding_layer = nn.Embedding(VOCAB_SIZE, EMBEDDING_DIM, padding_idx=word_to_index['<PAD>'])

# 2. Forward pass (Lookup)
input_indices = X_padded
dense_vectors = embedding_layer(input_indices)

print(f"\nInput Indices Shape: {input_indices.shape}")
print(f"Dense Vectors Shape: {dense_vectors.shape}")

# 3. Verification
pad_vector = dense_vectors[0, -1]
print(f"\nVector for PAD token (Sum of values): {pad_vector.sum().item()}")


Input Indices Shape: torch.Size([6, 8])
Dense Vectors Shape: torch.Size([6, 8, 100])

Vector for PAD token (Sum of values): 0.0


### Task 2: Building the LSTM Sequence Labeler
Implement a PyTorch class LSTMNERSegmenter that utilizes an LSTM for the sequence labeling task.

1. `Initialization (__init__)`:
  * Define the nn.Embedding layer (from Task 1).
  *  Define a Bidirectional LSTM layer (nn.LSTM).

    Input Size: Embedding dimension.
    Hidden Size: Choose a reasonable hidden dimension (e.g., 256).
    bidirectional=True: Essential for sequence labeling.
  
  * Define a final Linear Layer (nn.Linear) that maps the output of the Bi-LSTM (which is ) to the number of unique tags/labels.

2. `Forward Pass (forward)`:
* Pass the input indices through the embedding layer.
* Pass the embeddings into the Bi-LSTM.
* Pass the LSTM output through the final linear layer.

In [10]:
# Task-2
class LSTMNERSegmenter(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim, hidden_dim=256):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0) # padding_idx=0 corresponds to '<PAD>'

        # Bidirectional LSTM layer
        # input_size: embedding_dim
        # hidden_size: hidden_dim
        # num_layers: 1 (for simplicity, can be increased)
        # bidirectional=True for Bi-LSTM
        self.lstm = nn.LSTM(embedding_dim,
                            hidden_dim,
                            batch_first=True,
                            bidirectional=True)

        # Linear layer to map LSTM output to tag space
        # Since it's bidirectional, the output size is 2 * hidden_dim
        self.hidden2tag = nn.Linear(hidden_dim * 2, tagset_size)

    def forward(self, x):
        # x is a tensor of word indices (batch_size, seq_len)

        # 1. Pass input through embedding layer
        embedded = self.embedding(x) # (batch_size, seq_len, embedding_dim)

        # 2. Pass embeddings through Bi-LSTM
        # lstm_out: (batch_size, seq_len, hidden_dim * 2)
        # hidden: (num_layers * 2, batch_size, hidden_dim)
        # cell: (num_layers * 2, batch_size, hidden_dim)
        lstm_out, _ = self.lstm(embedded)

        # 3. Pass LSTM output through linear layer
        tag_logits = self.hidden2tag(lstm_out) # (batch_size, seq_len, tagset_size)

        return tag_logits

In [11]:
# Instantiation

# Hyperparameters
VOCAB_SIZE = len(word_to_index)
TAGSET_SIZE = len(tag_to_index)
EMBEDDING_DIM = 100
HIDDEN_DIM = 256

# Initialize the model
model = LSTMNERSegmenter(VOCAB_SIZE, TAGSET_SIZE, EMBEDDING_DIM, HIDDEN_DIM)

# Check architecture
print(model)


LSTMNERSegmenter(
  (embedding): Embedding(41, 100, padding_idx=0)
  (lstm): LSTM(100, 256, batch_first=True, bidirectional=True)
  (hidden2tag): Linear(in_features=512, out_features=7, bias=True)
)


### Task 3: Training and Evaluation

1. Define the `Loss Function` (e.g., nn.CrossEntropyLoss) and the Optimizer (e.g., optim.Adam).
2. **Implement the training loop:** iterate through batches of the dataset, perform the forward pass, calculate the loss, perform backpropagation (loss.backward()), and update the weights (optimizer.step()).
3. **Evaluation:** Calculate the `token-level` accuracy (the percentage of words correctly tagged). Briefly discuss why the more robust `F1-Score` is preferred over simple accuracy for the NER task, which involves class imbalance (many 'O' tags).


In [15]:
# Task-3
import torch.optim as optim

# 1. Hyperparameters
LEARNING_RATE = 0.01
EPOCHS = 100
TAG_PAD_IDX = tag_to_index['<PAD>']

# 2. Initialize Model, Loss, and Optimizer
loss_function = nn.CrossEntropyLoss(ignore_index=TAG_PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 3. Helper Function for Accuracy
def categorical_accuracy(preds, y, tag_pad_idx=0):
    max_preds = preds.argmax(dim=2, keepdim=True).squeeze(2)

    # Only consider non-padding elements
    non_pad_elements = (y != tag_pad_idx)

    correct = max_preds[non_pad_elements].eq(y[non_pad_elements])

    return correct.sum() / torch.FloatTensor([y[non_pad_elements].shape[0]])

In [16]:
# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
X_padded = X_padded.to(device)
y_padded = y_padded.to(device)

print(f"Training on {device}...")

model.train()
for epoch in range(EPOCHS):
    optimizer.zero_grad()

    # Forward pass
    predictions = model(X_padded)

    # Reshape for loss calculation
    loss = loss_function(predictions.view(-1, TAGSET_SIZE), y_padded.view(-1))

    # Calculate accuracy
    acc = categorical_accuracy(predictions, y_padded, TAG_PAD_IDX)

    # Backward pass and optimization
    loss.backward()
    optimizer.step()

    if (epoch+1) % 2 == 0:
        print(f"Epoch: {epoch+1:02} | Loss: {loss.item():.4f} | Accuracy: {acc.item()*100:.2f}%")


print("\n--- Evaluation ---")
model.eval()
with torch.no_grad():
    predictions = model(X_padded)
    eval_loss = loss_function(predictions.view(-1, TAGSET_SIZE), y_padded.view(-1))
    eval_acc = categorical_accuracy(predictions, y_padded, TAG_PAD_IDX)

    print(f"Final Evaluation | Loss: {eval_loss.item():.4f} | Accuracy: {eval_acc.item()*100:.2f}%")

Training on cpu...
Epoch: 02 | Loss: 0.0029 | Accuracy: 100.00%
Epoch: 04 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 06 | Loss: 0.0001 | Accuracy: 100.00%
Epoch: 08 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 10 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 12 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 14 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 16 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 18 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 20 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 22 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 24 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 26 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 28 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 30 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 32 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 34 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 36 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 38 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 40 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 42 | Loss: 0.0000 | Accuracy: 100.00%
Epoch: 44 | Loss: 0.0000 | Accuracy:

### Discussion: F1-Score vs. Accuracy

Token-level accuracy can be misleading in NER due to class imbalance, as the 'O' (Outside) tag is usually far more frequent. A model predicting 'O' for everything can achieve high accuracy while failing to identify any entities.

The F1-score (harmonic mean of precision and recall) is preferred because it balances false positives and false negatives, providing a more robust measure of a model's ability to correctly identify the less frequent, but more critical, named entities.

### Task 4: Architecture Comparison (Theoretical)

Briefly modify your code (or simply discuss the required modifications) to compare:

* `Simple RNN vs. LSTM vs. GRU.` Explain the conceptual difference between the Cell State (in LSTM) and the simple Hidden State (in RNN/GRU).

In [17]:
# Task-4: Output Snapshot and Theoretical Discussion

def decode_sequence(indices, idx_to_word, idx_to_tag):
    words = []
    tags = []
    for idx in indices:
        words.append(idx_to_word.get(idx.item(), '<UNK>'))
        tags.append(idx_to_tag.get(idx.item(), '<UNK>'))
    return words, tags

index_to_word = {idx: word for word, idx in word_to_index.items()}
index_to_tag = {idx: tag for tag, idx in tag_to_index.items()}

sample_input_indices = X_padded[0].cpu()
sample_true_labels_indices = y_padded[0].cpu()

model.eval()
with torch.no_grad():
    sample_predictions = model(sample_input_indices.unsqueeze(0).to(device))
    sample_predicted_labels_indices = sample_predictions.argmax(dim=2).squeeze(0).cpu()

def remove_padding(indices, pad_idx):
    return [idx for idx in indices if idx.item() != pad_idx]

non_padded_input_indices = remove_padding(sample_input_indices, word_to_index['<PAD>'])
non_padded_true_labels_indices = remove_padding(sample_true_labels_indices, tag_to_index['<PAD>'])
non_padded_predicted_labels_indices = remove_padding(sample_predicted_labels_indices, tag_to_index['<PAD>'])

sample_words_decoded, _ = decode_sequence(torch.tensor(non_padded_input_indices), index_to_word, index_to_tag)
_, sample_true_tags_decoded = decode_sequence(torch.tensor(non_padded_true_labels_indices), index_to_word, index_to_tag)
_, sample_predicted_tags_decoded = decode_sequence(torch.tensor(non_padded_predicted_labels_indices), index_to_word, index_to_tag)

print("--- Output Snapshot ---")
print("Sample Sentence:", " ".join(sample_words_decoded))
print("True Labels:    ", sample_true_tags_decoded)
print("Predicted Labels:", sample_predicted_tags_decoded)

--- Output Snapshot ---
Sample Sentence: Elon Musk is the CEO of Tesla
True Labels:     ['B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'B-ORG']
Predicted Labels: ['B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'B-ORG', 'O']


### Theoretical Comparison: RNN vs. LSTM vs. GRU
Your explanation is accurate. Here’s a slightly clearer and more structured version:

---

### Simple RNN vs. GRU vs. LSTM

#### 1. Simple RNN

Standard (vanilla) RNNs update their hidden state at each time step using:

$$
[
h_t = \tanh(W_h h_{t-1} + W_x x_t)
]
$$

Because gradients are repeatedly multiplied through many time steps during backpropagation, they tend to **vanish or explode**, making it difficult to learn **long-term dependencies**. As a result, simple RNNs struggle to remember information from far earlier in the sequence.

---

#### 2. GRU (Gated Recurrent Unit)

GRUs introduce **gating mechanisms** to control information flow:

* **Update gate (zₜ)** → decides how much past information to keep.
* **Reset gate (rₜ)** → decides how much past information to forget when computing new content.

Key characteristics:

* No separate cell state (only hidden state).
* Fewer parameters than LSTM.
* Computationally more efficient.
* Handles long-term dependencies much better than vanilla RNNs.

The update gate effectively creates a path that allows gradients to flow more easily across time steps, mitigating vanishing gradient issues.

---

#### 3. LSTM (Long Short-Term Memory)

LSTMs are more expressive and structured than GRUs. They introduce:

* **Forget gate** → decides what to remove from memory.
* **Input gate** → decides what new information to store.
* **Output gate** → decides what to expose as the hidden state.
* **Cell state (cₜ)** → separate long-term memory track.

The key innovation is the **cell state**, which acts like a memory conveyor belt. Because it is updated primarily through **additive operations**, gradients can flow more stably across long sequences.

Important distinction:

* **Cell state** → long-term memory
* **Hidden state** → short-term working memory / output at each step

This separation allows LSTMs to retain information over very long sequences more effectively than both vanilla RNNs and often GRUs.

---

### Summary Comparison

| Model | Gates | Separate Cell State | Long-Term Memory | Complexity |
| ----- | ----- | ------------------- | ---------------- | ---------- |
| RNN   | None  | No                  | Poor             | Low        |
| GRU   | 2     | No                  | Good             | Medium     |
| LSTM  | 3     | Yes                 | Very Good        | Higher     |

---

### Final Insight

* **Vanilla RNNs** → simple but unstable for long sequences.
* **GRUs** → efficient and strong in many practical tasks.
* **LSTMs** → more structured memory control and often better for very long dependencies.

Both GRUs and LSTMs were designed specifically to address the vanishing gradient problem, with LSTMs offering finer control via the explicit cell state.



### Bi-LSTM vs. Unidirectional LSTM for NER
A Unidirectional LSTM processes text sequentially from left to right (or right to left), meaning each token’s representation depends only on past context. While this works for many sequence tasks, it limits the model’s ability to use future information when making predictions.

A Bi-LSTM (Bidirectional LSTM), on the other hand, consists of two LSTMs:

One processes the sequence forward (left → right)

One processes it backward (right → left)

The final representation of each word is typically a concatenation of both forward and backward hidden states. This allows the model to incorporate both past and future context when predicting labels.